# Lab 06: Unsupervised Learning — K-Means and PCA
**Week 7 — October 26 · Due Monday, November 2 at 5:00 PM**

A fictional campus music app tracks how many hours each listener spends on 10 genres every week. Nobody has labeled the listeners, but the app wants to find groups with similar taste so it can build better playlists. The data are synthetic and do not describe real people.

By the end, you should be able to:
- Explain how unsupervised learning differs from supervised learning.
- Carry out the steps of k-means (Lloyd's algorithm) by hand.
- Choose a number of clusters with an elbow plot and interpret the clusters.
- Explain why k-means can give different answers from different starting points.
- Use PCA to reduce many columns to a few and measure how much information is kept.

Work from top to bottom. Run cells with **Shift + Enter**. Replace every **...** in an answer cell and complete the written responses. You can work with a neighbor and ask a tutor/TA for help.

Public Otter checks cover Q2–Q7. Written responses are reviewed separately. Keep the `tests/lab06/` folder beside the notebook and run the checks as you go.

## Setup

Keep **listener_profiles.csv** beside this notebook. Each row is one fictional listener. Every genre column (**pop**, **hip_hop**, **rnb**, **kpop**, **rock**, **indie**, **electronic**, **classical**, **jazz**, **lofi**) holds that listener's **hours per week** on that genre. **listener_id** is just a name tag.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

import otter

grader = otter.Notebook("Lab06.ipynb", tests_dir="tests/lab06")

listeners = pd.read_csv("listener_profiles.csv")
genres = ["pop", "hip_hop", "rnb", "kpop", "rock", "indie",
          "electronic", "classical", "jazz", "lofi"]
X = listeners[genres].to_numpy()

print("Listeners:", len(listeners), "| Genre columns:", len(genres))
listeners.head()

### Q1. What makes this unsupervised?

In Labs 03 and 05, every row had a **target** to predict. This table does not. Explain why this is an **unsupervised** learning problem, and name one way the app could use groups of similar listeners.

> Write your answer here.

## Part 1: K-means by hand

**K-means** splits the rows into **k** groups called **clusters**. Each cluster has a center called a **centroid**. Lloyd's algorithm finds them:

1. Choose **k** starting centroids.
2. **Assign** each point to its nearest centroid.
3. **Update** each centroid to the mean of the points assigned to it.
4. Repeat steps 2 and 3 until the assignments stop changing.

"Nearest" means the smallest straight-line (**Euclidean**) distance. For a point $x$ and a centroid $c$ with two features:

$$\text{distance} = \sqrt{(x_1 - c_1)^2 + (x_2 - c_2)^2}$$

To see what is happening, we start with just two columns, **lofi** and **electronic**, and **k = 3**. The black X marks are the starting centroids.

In [ ]:
X2 = listeners[["lofi", "electronic"]].to_numpy()
centroids = np.array([[2.0, 2.0], [6.0, 1.0], [1.0, 6.0]])

plt.scatter(X2[:, 0], X2[:, 1], s=12, alpha=0.5)
plt.scatter(centroids[:, 0], centroids[:, 1], marker="X", s=200, color="black", label="Starting centroids")
plt.xlabel("Lofi (hours per week)")
plt.ylabel("Electronic (hours per week)")
plt.legend()
plt.show()

### Q2. Assign each listener to a centroid

Complete the loop so that column **j** of **distances** holds every listener's distance to centroid **j**. Then save each listener's nearest centroid (0, 1, or 2) as **assignments**.

**Hints:** **X2 - centroids[j]** subtracts centroid **j** from every row. Square that, add across the two columns with **np.sum(..., axis=1)**, then take **np.sqrt**. **np.argmin(distances, axis=1)** gives the position of the smallest value in each row.

In [ ]:
distances = np.zeros((len(X2), 3))
for j in range(3):
    distances[:, j] = ...

assignments = ...
print("Listeners assigned to each centroid:", np.bincount(assignments))

In [ ]:
grader.check("q2")

### Q3. Update the centroids

Move each centroid to the **mean** of the listeners assigned to it. Save the result as **new_centroids** (one row per centroid).

**Hint:** **X2[assignments == j]** keeps only the rows assigned to centroid **j**. **.mean(axis=0)** averages each column.

In [ ]:
new_centroids = np.zeros_like(centroids)
for j in range(3):
    new_centroids[j] = ...
print(np.round(new_centroids, 2))

In [ ]:
grader.check("q3")

That was one round of Lloyd's algorithm. The next cell repeats the assign and update steps until the centroids stop moving, then plots the final clusters.

In [ ]:
current = centroids.copy()
for step in range(1, 21):
    dist = np.stack([np.sqrt(np.sum((X2 - c) ** 2, axis=1)) for c in current], axis=1)
    labels = np.argmin(dist, axis=1)
    updated = np.array([X2[labels == j].mean(axis=0) for j in range(3)])
    if np.allclose(updated, current):
        break
    current = updated
print("Centroids stopped moving after", step, "rounds")

plt.scatter(X2[:, 0], X2[:, 1], c=labels, s=12, cmap="tab10", alpha=0.6)
plt.scatter(current[:, 0], current[:, 1], marker="X", s=200, color="black", label="Final centroids")
plt.xlabel("Lofi (hours per week)")
plt.ylabel("Electronic (hours per week)")
plt.legend()
plt.show()

**Your response:** Describe the three clusters in words. The biggest cluster contains listeners who play little lofi **and** little electronic music. Why might those listeners still have very different tastes?

> Write your answer here.

## Part 2: K-means on all 10 genres

Lloyd's algorithm works the same way with 10 columns; we just cannot draw it. From here on we use scikit-learn's **KMeans** on all genres in **X**.

To compare clusterings, k-means measures **inertia**: the sum of squared distances from each point to its own centroid. Lower inertia means tighter clusters. But the optimal inertia **never increases** when you add clusters: with one cluster per listener it would be 0. So instead of choosing the k with the lowest inertia, we look for the **elbow**, the point where adding another cluster stops helping much.

We intentionally compare **absolute weekly hours** here: every genre uses the same unit and a similar range. Standardizing would instead give each genre equal variance and could change the groups. Even columns with the same units may need scaling if their ranges differ greatly. Inertia is non-increasing at the optimum as k grows; individual fitted runs can occasionally miss that optimum.

### Q4. Make an elbow plot

For each **k** from 1 to 8, fit a **KMeans** model on **X** and append its **inertia_** to **inertias**.

In [ ]:
ks = range(1, 9)
inertias = []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    km.fit(...)
    inertias.append(...)

plt.plot(ks, inertias, "o-")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia")
plt.title("Elbow plot")
plt.show()

In [ ]:
grader.check("q4")

**Your response:** Which k does the elbow suggest? Why would picking the k with the **lowest** inertia be a bad idea?

> Write your answer here.

### Q5. Fit and name the clusters

Fit **kmeans** with the number of clusters you chose. Save each listener's cluster number (**kmeans.labels_**) in a new **cluster** column. Then save the average hours per genre for each cluster as **cluster_means**.

**Hint:** Group by **cluster**, select the **genres** columns, and take the mean.

In [ ]:
kmeans = KMeans(n_clusters=..., n_init=10, random_state=42).fit(X)
listeners["cluster"] = ...
cluster_means = ...

print("Listeners per cluster:", listeners["cluster"].value_counts().sort_index().tolist())
cluster_means.round(1)

In [ ]:
grader.check("q5")

**Your response:** Give each cluster a short name based on its top genres (for example, "late-night lofi"). The cluster numbers themselves mean nothing; why not?

> Write your answer here.

### Different starts, different answers

So far we used **n_init=10**: scikit-learn ran k-means from 10 different starting points and kept the result with the lowest inertia. The next cell runs k-means **once** from each of six random starts.

In [ ]:
for seed in range(6):
    km = KMeans(n_clusters=4, init="random", n_init=1, random_state=seed).fit(X)
    sizes = sorted(np.bincount(km.labels_).tolist(), reverse=True)
    print(f"Start {seed}: inertia = {km.inertia_:,.0f} | cluster sizes = {sizes}")

**Your response:** Why do some starts end with a much higher inertia? Which result would you trust, and how does **n_init** help?

> Write your answer here.

## Part 3: PCA and dimensionality reduction

We cannot plot 10 columns, and many columns move together anyway (lofi listeners also tend to play classical). **Principal component analysis (PCA)** builds new axes called **principal components**:

- **PC1** points in the direction where listeners differ the most.
- **PC2** captures the most remaining spread, at a right angle to PC1, and so on.
- Each component is a **weighted mix** of the original genre columns.

Keeping only the first few components is **dimensionality reduction**: fewer columns that still hold most of the variance. The **explained variance ratio** tells us what share of the total spread each component captures.

PCA and k-means both rely on distances and spread. Our columns all share one unit (hours per week). If they did not, for example minutes next to counts, we would **standardize** them first so that no column dominates just because of its units. Equal units alone do not guarantee equal ranges; inspect both and choose a distance definition that fits the task. PCA preserves variance, which is not necessarily the information most useful for a later prediction task.

### Q6. How many components do we need?

Fit PCA on **X** (provided). Save the share of variance captured by each component as **explained**, the running total as **cumulative**, and the smallest number of components that reaches **80%** as **n_for_80**.

**Hints:** Use **pca.explained_variance_ratio_** and **np.cumsum**. **np.argmax(cumulative >= 0.80)** gives the *position* of the first component that reaches 80%; positions start at 0, so add 1.

In [ ]:
pca = PCA().fit(X)
explained = ...
cumulative = ...
n_for_80 = ...

print("Share of variance per component:", np.round(explained, 3))
print("Components needed for 80%:", n_for_80)

In [ ]:
grader.check("q6")

In [ ]:
components = np.arange(1, len(explained) + 1)
plt.bar(components, explained, label="Each component")
plt.plot(components, cumulative, "o-", color="tab:orange", label="Running total")
plt.axhline(0.80, color="gray", linestyle="--", label="80%")
plt.xlabel("Principal component")
plt.ylabel("Share of variance explained")
plt.xticks(components)
plt.legend()
plt.show()

### Q7. See the clusters in 2D

Reduce **X** to **2** components and save the result as **X_2d**. The provided lines then plot every listener, colored by their k-means cluster, and show each component's weights on the genres.

**Hint:** Create **PCA(n_components=...)**, then call **.fit_transform(X)**.

In [ ]:
pca_2d = PCA(n_components=...)
X_2d = ...

for c in sorted(listeners["cluster"].unique()):
    in_cluster = (listeners["cluster"] == c).to_numpy()
    plt.scatter(X_2d[in_cluster, 0], X_2d[in_cluster, 1], s=12, alpha=0.7, label=f"Cluster {c}")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Listeners in 2 dimensions, colored by k-means cluster")
plt.legend()
plt.show()

print("Share of variance kept in 2D:", round(pca_2d.explained_variance_ratio_.sum(), 3))
pd.DataFrame(pca_2d.components_, index=["PC1", "PC2"], columns=genres).round(2)

In [ ]:
grader.check("q7")

**Your response:**
1. Look at the weights table. Which genres have the largest positive weights on **PC2**? What kind of listener gets a high PC2 score?
2. Two of the clusters overlap in the 2D plot. Which ones? K-means separated them using all 10 columns, so why can they overlap here?

A component’s overall sign can flip across implementations. Interpret the displayed weights and scores together; the underlying axis is the same.

> Write your answer here.

## Wrap-up

In 2–3 sentences: k-means and PCA are both unsupervised, but they simplify the data in different ways. What does each one reduce, and how did they work together in this lab?

> Write your answer here.

In [ ]:
grader.check_all()